# FedAI — Federal Reserve RAG Notebook (Gemini API)
A modernized, notebook-first rework of the CentralBank-LLM app (Fed-only)

**What this notebook does**
- Downloads recent FOMC Minutes and FOMC Statements PDFs from the Federal Reserve.
- Extracts & chunks text with metadata (date, source).
- Builds a semantic index (FAISS or sklearn fallback) using Gemini embeddings.
- Runs Retrieval-Augmented Generation (RAG) with Gemini for grounded answers + citations.

**Notes**
- All code is notebook-native (no Dash/Plotly UI).
- Uses the Google GenAI SDK (`google-genai`) with models:
  - Text generation: `gemini-2.5-flash` (price-performance, thinking-enabled)
  - Embeddings: `gemini-embedding-001` (supports configurable output dimensions)
- You can adjust paths and parameters in the Config cell.
- Created on 2025-09-16.


## 0) Setup
> Run once per environment. If you don't have these packages, uncomment and install.


In [ ]:

# %% Optional installs (uncomment if needed)
# %pip install -U google-genai pypdf numpy faiss-cpu scikit-learn tiktoken

import os, re, sys, math, time, json, uuid, pathlib, typing, datetime, itertools, textwrap
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

# Core deps
import numpy as np

# PDF parsing
from pypdf import PdfReader

# Vector index: try FAISS first; fallback to sklearn
try:
    import faiss  # faiss-cpu recommended in pip install
    HAVE_FAISS = True
except Exception:
    HAVE_FAISS = False
    from sklearn.neighbors import NearestNeighbors

# Gemini API (Google GenAI SDK)
from google import genai
from google.genai import types as genai_types


## 1) Config

In [ ]:

# Paths (you can change these)
DATA_DIR = pathlib.Path("./fed_data").resolve()
PDF_DIR = DATA_DIR / "pdfs"
INDEX_DIR = DATA_DIR / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

# Download window
DAYS_BACK = 365  # how many days to scan backwards for FOMC minutes/statements

# Embeddings config
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768   # smaller size to save space (supported by gemini-embedding-001)
EMBED_TASK_DOC = "RETRIEVAL_DOCUMENT"
EMBED_TASK_QUERY = "QUESTION_ANSWERING"

# Generation model
GEN_MODEL = "gemini-2.5-flash"

# RAG defaults
TOP_K = 8
CHUNK_SIZE = 1200       # characters
CHUNK_OVERLAP = 240     # characters
MAX_CHUNKS_PER_DOC = 99999

# Rate limiting
SLEEP_BETWEEN_REQUESTS = 0.15  # seconds

# API key: set via env var for security (recommended)
# os.environ["GEMINI_API_KEY"] = "YOUR_KEY_HERE"  # or set in the cell below


### 1.a) Provide your GEMINI_API_KEY (one-time per session)

In [ ]:

import getpass, os
if "GEMINI_API_KEY" not in os.environ or not os.environ["GEMINI_API_KEY"]:
    print("Enter your GEMINI_API_KEY (input hidden):")
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
else:
    print("Using GEMINI_API_KEY from environment")
    
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Gemini client initialized.")


## 2) Data ingestion — Federal Reserve PDFs

This replicates the original app’s logic for the Federal Reserve only, scanning the last `DAYS_BACK` days for files under:
```
https://www.federalreserve.gov/monetarypolicy/files/
  - fomcminutesYYYYMMDD.pdf
  - monetaryYYYYMMDDa1.pdf
```


In [ ]:

import requests
from datetime import datetime, timedelta
from typing import List, Tuple, Optional, Dict

FED_BASE = "https://www.federalreserve.gov/monetarypolicy/files/"

def fed_candidate_urls(day: datetime) -> List[Tuple[str,str]]:
    y = day.strftime("%Y"); m = day.strftime("%m"); d = day.strftime("%d")
    # (filename, full_url)
    return [
        (f"fomcminutes{y}{m}{d}.pdf", f"{FED_BASE}fomcminutes{y}{m}{d}.pdf"),
        (f"monetary{y}{m}{d}a1.pdf",  f"{FED_BASE}monetary{y}{m}{d}a1.pdf"),
    ]

def safe_get(url: str, timeout=15) -> Optional[bytes]:
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and r.headers.get("Content-Type","").lower().startswith("application/pdf"):
            return r.content
        # Some Fed servers return 200 with non-pdf (rare). Guard below:
        if r.status_code == 200 and url.lower().endswith(".pdf"):
            return r.content  # still save; PDF parsers will validate
    except Exception:
        return None
    return None

def download_fed_pdfs(days_back: int = DAYS_BACK, out_dir: pathlib.Path = PDF_DIR) -> Dict[str, dict]:
    """
    Returns dict: {filepath: {"date": iso_date, "author": "FOMC Minutes|FOMC Statement"}}
    """
    meta = {}
    out_dir.mkdir(parents=True, exist_ok=True)
    today = datetime.utcnow().date()
    for k in range(days_back):
        day = today - timedelta(days=k)
        for fname, url in fed_candidate_urls(day):
            payload = safe_get(url)
            if not payload:
                continue
            fpath = out_dir / fname
            with open(fpath, "wb") as f:
                f.write(payload)
            author = "FOMC Minutes" if fname.startswith("fomcminutes") else "FOMC Statement"
            # Use month granularity for charts (consistent with original)
            doc_date = datetime(day.year, day.month, 1).isoformat()
            meta[str(fpath)] = {"date": doc_date, "author": author}
            print("Downloaded:", fname)
            time.sleep(0.05)
        # be polite
        time.sleep(0.02)
    return meta

# Run the downloader (you can skip to use pre-downloaded PDFs)
# meta = download_fed_pdfs()
# len(meta)


## 3) Parse PDFs -> Text chunks with metadata

In [ ]:

from typing import List

def read_pdf_text(pdf_path: pathlib.Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for i, page in enumerate(reader.pages):
        try:
            parts.append(page.extract_text() or "")
        except Exception:
            parts.append("")
    return "\n".join(parts)

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text:
        return []
    chunks = []
    start = 0
    n = len(text)
    while start < n and len(chunks) < MAX_CHUNKS_PER_DOC:
        end = min(n, start + chunk_size)
        # try not to break mid-sentence if possible
        slice_ = text[start:end]
        if end < n:
            # find last period/newline within the last 120 chars
            backtrack = slice_.rfind(".")
            backtrack_nl = slice_.rfind("\n")
            bt = max(backtrack, backtrack_nl)
            if bt > 0 and (end - (start + bt)) < 120:
                end = start + bt + 1
                slice_ = text[start:end]
        chunks.append(slice_)
        start = max(end - overlap, start + 1)
    return [c.strip() for c in chunks if c.strip()]

def build_corpus(meta: Dict[str, dict], pdf_dir: pathlib.Path = PDF_DIR) -> List[dict]:
    """
    Returns list of {'id', 'text', 'date', 'author', 'source'}
    """
    rows = []
    for fpath_str, m in meta.items():
        fpath = pathlib.Path(fpath_str)
        text = read_pdf_text(fpath)
        for i, ch in enumerate(chunk_text(text)):
            rows.append({
                "id": f"{fpath.name}::chunk{i:04d}",
                "text": ch,
                "date": m.get("date"),
                "author": m.get("author"),
                "source": fpath.name,
            })
    return rows

# Example:
# meta = download_fed_pdfs()
# corpus = build_corpus(meta)
# len(corpus), corpus[0]


## 4) Embeddings & Vector Index (FAISS or sklearn fallback)

In [ ]:

from typing import Optional

from dataclasses import dataclass
import numpy as np

@dataclass
class IndexArtifacts:
    dim: int
    ids: List[str]
    meta: List[dict]
    # FAISS index saved separately; for sklearn we keep model in memory only.
    faiss_path: Optional[str] = None
    vectors_path: Optional[str] = None
    idmap_path: Optional[str] = None
    meta_path: Optional[str] = None

def embed_texts(client: genai.Client, texts: List[str], task_type: str = EMBED_TASK_DOC, dim: int = EMBED_DIM, batch_size: int = 64) -> np.ndarray:
    """
    Returns ndarray [n, dim]
    """
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.models.embed_content(
            model=EMBED_MODEL,
            contents=batch if len(batch) > 1 else batch[0],
            config=genai_types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=dim
            )
        )
        # API returns .embeddings as a list of objects with .values
        vecs = [np.array(e.values, dtype=np.float32) for e in resp.embeddings]
        out.extend(vecs)
        time.sleep(SLEEP_BETWEEN_REQUESTS)
    return np.vstack(out).astype(np.float32)

def build_index(client: genai.Client, corpus: List[dict], index_dir: pathlib.Path = INDEX_DIR, dim: int = EMBED_DIM) -> IndexArtifacts:
    texts = [r["text"] for r in corpus]
    ids = [r["id"] for r in corpus]
    meta = [{"date": r["date"], "author": r["author"], "source": r["source"]} for r in corpus]
    vectors = embed_texts(client, texts, task_type=EMBED_TASK_DOC, dim=dim)

    idmap = {i: ids[i] for i in range(len(ids))}

    # save metadata & vectors
    vectors_path = str(index_dir / "doc_vectors.npy")
    idmap_path = str(index_dir / "idmap.json")
    meta_path = str(index_dir / "meta.json")
    np.save(vectors_path, vectors)
    json.dump(idmap, open(idmap_path, "w"))
    json.dump(meta, open(meta_path, "w"))

    if HAVE_FAISS:
        index = faiss.IndexFlatIP(dim)  # cosine via inner product if normalized
        # Normalize to unit length for cosine similarity
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        v_unit = vectors / norms
        index.add(v_unit)
        faiss_path = str(index_dir / "faiss.index")
        faiss.write_index(index, faiss_path)
        return IndexArtifacts(dim=dim, ids=ids, meta=meta, faiss_path=faiss_path, vectors_path=vectors_path, idmap_path=idmap_path, meta_path=meta_path)
    else:
        # sklearn fallback
        nn = NearestNeighbors(n_neighbors=min(20, len(vectors)), metric="cosine")
        nn.fit(vectors)
        # serialize fallback with joblib if available; otherwise keep in memory
        try:
            import joblib
            joblib.dump(nn, index_dir / "nn.joblib")
        except Exception:
            pass
        return IndexArtifacts(dim=dim, ids=ids, meta=meta, faiss_path=None, vectors_path=vectors_path, idmap_path=idmap_path, meta_path=meta_path)

def load_index(index_dir: pathlib.Path = INDEX_DIR):
    vectors = np.load(index_dir / "doc_vectors.npy")
    idmap = json.load(open(index_dir / "idmap.json"))
    meta = json.load(open(index_dir / "meta.json"))
    inv_id = {int(k): v for k, v in idmap.items()}
    artifacts = IndexArtifacts(dim=vectors.shape[1], ids=[inv_id[i] for i in range(len(inv_id))], meta=meta)
    model = None
    if HAVE_FAISS and (index_dir / "faiss.index").exists():
        model = faiss.read_index(str(index_dir / "faiss.index"))
    else:
        try:
            import joblib
            model = joblib.load(index_dir / "nn.joblib")
        except Exception:
            model = None
    return artifacts, model, vectors, inv_id, meta

def search(client: genai.Client, query: str, k: int = TOP_K, index_dir: pathlib.Path = INDEX_DIR) -> List[dict]:
    artifacts, model, vectors, inv_id, meta = load_index(index_dir)
    q = embed_texts(client, [query], task_type=EMBED_TASK_QUERY, dim=artifacts.dim)[0]
    # cosine similarity search
    if HAVE_FAISS and model is not None:
        # faiss index expects normalized vectors for IP
        q = q / (np.linalg.norm(q) + 1e-12)
        D, I = model.search(q.reshape(1, -1), k)
        idxs = I[0].tolist()
        scores = D[0].tolist()
    else:
        # brute force if no index
        from sklearn.metrics.pairwise import cosine_similarity
        sims = cosine_similarity([q], vectors)[0]
        idxs = np.argsort(-sims)[:k].tolist()
        scores = sims[idxs].tolist()
    results = []
    for row_idx, score in zip(idxs, scores):
        doc_id = inv_id[row_idx]
        m = meta[row_idx]
        results.append({"id": doc_id, "score": float(score), **m})
    return results


## 5) RAG: generate grounded answers with citations

In [ ]:

from typing import Optional

CITATION_SEPARATOR = " /// "

SYSTEM_PROMPT = """You are an expert financial research assistant specializing in U.S. Federal Reserve publications.
Always base your answers STRICTLY on the provided retrieved context, which comes from FOMC Minutes and FOMC Statements.
Return a concise, structured answer and include a "Sources" section listing source filename and month (YYYY-MM).
If the user asks for forecasts or outlook, summarize only what is stated explicitly in the retrieved text.

Response format:
- Short answer (2-5 sentences)
- 3-7 bullet highlights (facts, rationale, dates)
- Sources: each on a new line as FILE (YYYY-MM)

If a question cannot be answered from the context, say so and suggest a relevant query."""

def load_chunk_text_by_id(doc_id: str, pdf_dir: pathlib.Path = PDF_DIR) -> Optional[str]:
    # doc_id looks like "fomcminutesYYYYMMDD.pdf::chunk0007"
    fname, _, chunk_str = doc_id.partition("::chunk")
    fpath = pdf_dir / fname
    if not fpath.exists():
        return None
    text = read_pdf_text(fpath)
    chunks = chunk_text(text)
    try:
        idx = int(chunk_str)
        return chunks[idx]
    except Exception:
        return None

def gather_context(results: List[dict], max_chars: int = 8000) -> str:
    parts = []
    total = 0
    for r in results:
        chunk = load_chunk_text_by_id(r["id"])
        if not chunk:
            continue
        header = f"[{r['source']} | {r['author']} | {r['date']}]"
        block = header + "\n" + chunk
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts)

def answer_query(client: genai.Client, question: str, k: int = TOP_K) -> dict:
    hits = search(client, question, k=k)
    context = gather_context(hits)
    if not context.strip():
        answer = "I could not find relevant context in the indexed documents. Try broadening the time window or re-indexing."
        return {"answer": answer, "sources": [], "raw": None, "hits": hits}
    
    sys_inst = genai_types.Content(role="system", parts=[genai_types.Part.from_text(SYSTEM_PROMPT)])
    user_msg = genai_types.Content(role="user", parts=[genai_types.Part.from_text(f"Context:\n{context}\n\nQuestion:\n{question}")])

    resp = client.models.generate_content(
        model=GEN_MODEL,
        contents=[sys_inst, user_msg],
        config=genai_types.GenerateContentConfig(
            temperature=0.3,
            max_output_tokens=1200,
        ),
    )
    text = getattr(resp, "text", None) or ""
    srcs = []
    for r in hits:
        ym = (r["date"] or "")[:7]
        srcs.append(f"{r['source']} ({ym})")
    return {"answer": text, "sources": srcs, "raw": resp, "hits": hits}


## 6) End-to-end: Build / Search / Generate

Run these cells in order. The downloader can take a few minutes depending on network and rate limits. You may also manually place PDFs into `./fed_data/pdfs` and skip the download step.


In [ ]:

# 6.1) Download (or skip if PDFs already present)
# meta = download_fed_pdfs(DAYS_BACK, PDF_DIR)

# 6.2) If you already have PDFs, reconstruct meta from filenames as "unknown" author/date (less accurate)
def reconstruct_meta_from_folder(pdf_dir: pathlib.Path = PDF_DIR) -> Dict[str, dict]:
    meta = {}
    for f in pdf_dir.glob("*.pdf"):
        if f.name.startswith("fomcminutes"):
            author = "FOMC Minutes"
        elif f.name.startswith("monetary"):
            author = "FOMC Statement"
        else:
            author = "Federal Reserve"
        import re, datetime as dt
        m = re.search(r"(\d{4})(\d{2})(\d{2})", f.name)
        if m:
            y, mo, d = m.groups()
            date_iso = dt.datetime(int(y), int(mo), 1).isoformat()
        else:
            date_iso = dt.datetime(2000,1,1).isoformat()
        meta[str(f.resolve())] = {"date": date_iso, "author": author}
    return meta

# meta = reconstruct_meta_from_folder()

# 6.3) Build corpus & index
# corpus = build_corpus(meta, PDF_DIR)
# artifacts = build_index(client, corpus, INDEX_DIR, dim=EMBED_DIM)

# 6.4) Query
# question = "Summarize how the FOMC characterized inflation and labor market conditions in the latest minutes."
# out = answer_query(client, question, k=TOP_K)
# print(out["answer"])
# print("\nSources:\n" + "\n".join(out["sources"]))


## 7) (Optional) Simple evaluation stubs

These are lightweight utilities you can adapt:
- Recall@k: Check whether gold snippets are retrieved among top-k. (Requires labeled pairs.)
- Self-check: Ask the model to cite which retrieved chunks support each bullet.


In [ ]:

def recall_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int = 8) -> float:
    """
    Proportion of gold ids appearing in top-k retrieved_ids.
    """
    topk = set(retrieved_ids[:k])
    gold = set(gold_ids)
    if not gold:
        return float("nan")
    return len(topk & gold) / len(gold)

# Example usage (requires you to supply gold_ids):
# hits = search(client, "What did the FOMC say about inflation risks in 2025?")
# r_ids = [h["id"] for h in hits]
# recall_at_k(r_ids, gold_ids, k=8)


## 8) Save / Load tips

- Index files are stored in `./fed_data/index/`:
  - `doc_vectors.npy` — dense vectors
  - `idmap.json` — row index -> chunk id
  - `meta.json` — metadata aligned with vectors
  - `faiss.index` or `nn.joblib` — index structure (depending on availability)

In [ ]:

print("Notebook initialized. Edit config, then run sections 6.1 -> 6.4 to use the pipeline.")
